# 개별종목 조합B — LogisticRegression

`기본모델/01.LogisticRegression.ipynb`과 같은 `models.logistic.build_logistic_baseline`을 가져오고
조합B 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.logistic import build_logistic_baseline  # noqa: E402

MODEL_NAME = 'LogisticRegression'
MODEL_BUILDER = build_logistic_baseline


In [2]:
# 2. 조합B의 피처 값만 지정합니다.
import json

COMBINATION = 'B'
FEATURE_COLUMNS = (
    'ret_5',
    'ret_20',
    'sma_gap_5_20',
    'sma_gap_20_60',
    'rsi_14',
    'dist_high_20',
    'dist_high_60',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
combination_report = report["combinations"][COMBINATION]
panel = combination_report["panel"]
print("학습 기간:", panel["first_date"], "~", panel["last_date"])
print("학습 행·종목:", panel["model_rows"], panel["stocks"])
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)

folds = pd.DataFrame(combination_report["outer_fold_results"])
model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
fold_columns = [
    "fold",
    "selected_class_weight",
    "train_dates",
    "valid_start",
    "valid_end",
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, fold_columns].round(4))

metric_columns = [
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 24개 노트북이 각각 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


학습 기간: 20100331 ~ 20240822
학습 행·종목: 172884 162
조합B 피처: ('ret_5', 'ret_20', 'sma_gap_5_20', 'sma_gap_20_60', 'rsi_14', 'dist_high_20', 'dist_high_60')


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,down_recall,core_harmonic_mean
0,1,balanced,750,20130410,20130705,0.3993,0.3701,0.0292,0.3705,0.2177,0.3062
1,2,balanced,999,20140414,20140711,0.4531,0.4743,-0.0212,0.3397,0.1718,0.2735
2,3,balanced,1248,20150421,20150716,0.3679,0.3301,0.0378,0.3675,0.3039,0.3437
3,4,balanced,1496,20160422,20160719,0.4147,0.4107,0.0040,0.3858,0.2457,0.3307
4,5,balanced,1745,20170424,20170721,0.3707,0.4177,-0.0470,0.3464,0.3731,0.3630
5,6,balanced,1994,20180503,20180731,0.3937,0.3907,0.0031,0.3895,0.2887,0.3500
6,7,balanced,2243,20190514,20190806,0.4027,0.4612,-0.0585,0.3471,0.1466,0.2462
7,8,balanced,2492,20200518,20200807,0.3532,0.3144,0.0388,0.3452,0.5194,0.3920
8,9,balanced,2741,20210518,20210810,0.4076,0.4418,-0.0342,0.3769,0.2815,0.3465
9,10,balanced,2989,20220519,20220812,0.3840,0.3343,0.0497,0.3662,0.1657,0.2638


,OOS 폴드 평균
accuracy,0.3932
training_majority_baseline_accuracy,0.3846
accuracy_minus_training_majority_baseline,0.0087
macro_f1,0.3657
down_recall,0.2716
core_harmonic_mean,0.3240


재실행 명령: python scripts/run_stock_model_experiment.py
